In [ ]:
# Global Oil and Gas Exposure Study
# 01b: PopExposure Analysis

# This file applies Popexposure to finds the number of people living near buffered oil and gas wells. 
# Author: Lara Schwarz
# Last updated: October 2025
# Reviewed by Nina Flores: September 2025

## Function step 1: Estimating population affected by oil and gas wells - country level and adm2
## Function step 3: Estimating population affected by oil and gas wells by adm2 - out of country wells
## Function step 4: Estimating population affected by oil and gas wells by adm2 - offshore wells
## Function step 5: Estimating total area exposed by country and by adm2
## Function step 6: Estimating ogd well-weighted population exposure 
## Function step 7: Estimating average GRDI by adm2

In [1]:
## Importing libraries

import geopandas as gpd
import pandas as pd
import pathlib
import matplotlib.pyplot as plt
from thefuzz import process
import pandas as pd
import pathlib
import pandas as pd
import sys
import os
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
import geopandas as gpd
from rasterio.plot import show
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import fiona
import geopandas as gpd
from shapely.geometry import Polygon
import ee
import geemap
import matplotlib.cm as cm
import matplotlib.colors as colors
from shapely.geometry import box
import math
from shapely.geometry import Point
import glob
import seaborn as sns
import openpyxl
import subprocess
from pathlib import Path
from popexposure import PopEstimator



/opt/anaconda3/envs/pop_exp/lib/python3.11/site-packages/geemap/conversion.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
### Prep and setup

# Define base paths
repo = pathlib.Path.cwd().parent.parent
share_path = pathlib.Path("/Users/larasch/Library/CloudStorage/OneDrive-SharedLibraries-UW/casey_cohort - Documents/data")
gee_path= pathlib.Path("Users/larasch/Library/CloudStorage/GoogleDrive-laranschwarz@gmail.com/My Drive/EarthEngineExports")

# Define code path and add to sys.path
code_path = repo / "code"

# Define data paths
# location of the oil and gas data
oil_path = share_path / "environmental/oil_and_gas/raw_data/ogim.parquet"

# location of global shapefile
shapefile_path = repo / "data/OneDrive_1_12-11-2024/lbd_standard_admin_2.shp"

# location of the population data
pop_path = share_path / "social_including_census/raw_data/GHS_POP_E2020_GLOBE_R2023A_54009_100_V1_0/GHS_POP_E2020_GLOBE_R2023A_54009_100_V1_0.tif"

# Define the directory for the country-specific parquet files
country_dir = share_path / "geo_boundaries/processed_data/countries_ogd"

# Define the directory for the OGIM country-specific parquet files
ogim_dir = share_path / "environmental/oil_and_gas/raw_data/ogim_by_country"

# Define the directory for results
results_dir = repo / "results"

# Import local modules
from popexposure import *

# Use estimator
pop_est = PopEstimator()

# Define dataset for land exposure
#land_path = gee_path / "MOD44W_2015_land_mask_clipped_anyoverlap.tif"
land_path = repo / "data/MOD44W_2015_land_mask_clipped_anyoverlap.tif"

# Define countries to loop through
country_list = [
    "argentina", "australia", "austria", "belgium", "bolivia", "brazil", "canada", "chile", 
    "colombia", "denmark", "ecuador", "ethiopia", "france", "germany", "guatemala", "ireland", "italy", 
    "libya", "mexico", "mozambique", "netherlands", "new_zealand", "norway", 
    "papua_new_guinea", "paraguay", "peru", "saudi_arabia", "south_sudan", "sudan", 
    "united_arab_emirates", "united_kingdom", "united_states", "venezuela"
]


In [ ]:
# Function step 1: Estimating population affected by oil and gas wells by adm2 and country

## Apply Popexposure function
results = {}

# Find total num ppl residing 
exposed_list = []
pop_list = []

# Loop through each country
for country in country_list:
    print(f"Processing {country}...")

    # types of wells
    hazard_paths = ogim_dir / f"OGIM_{country}.parquet"
    
    # country parquet
    admin_units_path = country_dir / f"{country}_adm2.parquet"

    # Prepare admin units data
    admin_units = pop_est.prep_data(admin_units_path, geo_type='admin_unit')

    
    for hazard_path in [hazard_paths]:
        try:
            # Prepare hazard data for this year
            hazards = pop_est.prep_data(hazard_path, geo_type='hazard')

            exposed = pop_est.est_exposed_pop(
                pop_path=pop_path,
                hazard_specific=False,  # set to True if you want per-hazard results
                hazards=hazards,
                admin_units=admin_units
            )

        except KeyError as e:
            print(f"  Skipping {country}: missing key {e}.")
            continue
        except Exception as e:
            print(f"  Skipping {country}: unexpected error {e}")
            continue

        # Skip if no result
        if exposed is None or getattr(exposed, "empty", True):
            print(f"  Skipping {country}: exposure result is empty or invalid.")
            continue

        exposed["country"] = country  
        exposed_list.append(exposed)

# Estimate total population per spatial unit for this country
    pop_df = pop_est.est_pop(pop_path=pop_path, admin_units=admin_units)
    pop_df["country"] = country  # Optionally add country column for tracking
    pop_list.append(pop_df)      # <-- Collect each country's pop_df

# Combine all countries
all_exposed_df = pd.concat(exposed_list, axis=0, ignore_index=True)
all_pop_df = pd.concat(pop_list, axis=0, ignore_index=True)

# Merge exposed and total population on ID_spatial_unit and country
combined_df = all_pop_df.merge(
    all_exposed_df[["ID_admin_unit", "exposed_1km", "exposed_3km","exposed_5km", "country"]],
    on=["ID_admin_unit", "country"],
    how="left"
)

# Add percent exposed columns
combined_df["pct_exposed_1km"] = combined_df["exposed_1km"] / combined_df["population"]
combined_df["pct_exposed_3km"] = combined_df["exposed_3km"] / combined_df["population"]
combined_df["pct_exposed_5km"] = combined_df["exposed_5km"] / combined_df["population"]

# Optionally, handle division by zero or NaN
combined_df["pct_exposed_1km"] = combined_df["pct_exposed_1km"].replace([np.inf, -np.inf], np.nan)
combined_df["pct_exposed_3km"] = combined_df["pct_exposed_3km"].replace([np.inf, -np.inf], np.nan)
combined_df["pct_exposed_5km"] = combined_df["pct_exposed_5km"].replace([np.inf, -np.inf], np.nan)


# Save output
output_path = results_dir / "ogd_adm2_pop_exposed.csv"
combined_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")


country_summary = (
    combined_df.groupby("country", as_index=False)
    .agg({
        "exposed_1km": "sum",
        "exposed_3km": "sum",
        "exposed_5km": "sum",
        "population": "sum"
    })
)

# Compute country-level percentages
for dist in ["1km", "3km", "5km"]:
    country_summary[f"pct_exposed_{dist}"] = (
        country_summary[f"exposed_{dist}"] / country_summary["population"]
    )

# Handle NaNs and infinite values
country_summary.replace([np.inf, -np.inf], np.nan, inplace=True)

# Export to CSV
country_output_path = results_dir / "ogd_country_pop_exposed.csv"
country_summary.to_csv(country_output_path, index=False)
print(f"Saved country-level summary: {country_output_path}")


In [ ]:
# Function step 3: Estimating population affected by oil and gas wells by adm2 - out of country wells

## Apply Popexposure function
results = {}

exposed_list = []
pop_list = []

for country in country_list:
    print(f"Processing {country}...")

    hazard_path = ogim_dir / f"OGIM_{country}_out_of_country_modis.parquet"
    admin_units_path = country_dir / f"{country}_adm2.parquet"

    if not hazard_path.exists():
        print(f"  Skipping {country}: out of country file not found.")
        continue
    if not admin_units_path.exists():
        print(f"  Skipping {country}: admin2 file not found.")
        continue

    admin_units = pop_est.prep_data(admin_units_path, geo_type='admin_unit')
    hazards = pop_est.prep_data(hazard_path, geo_type='hazard')

    # Skip if hazards is None or empty
    if hazards is None or getattr(hazards, "empty", True):
        print(f"  Skipping {country}: hazards file is empty or unreadable.")
        continue

    # Restore well_country if it was dropped by prep_data 
    if "well_country" not in hazards.columns: 
        original = pd.read_parquet(hazard_path, columns=["well_country"]) 
        hazards["well_country"] = original["well_country"].values

    # Subset by well_country if present
    if "well_country" in hazards.columns:
        for well_country, hazards_group in hazards.groupby("well_country"):
            if hazards_group.empty:
                print(f"    Skipping well_country {well_country}: hazards group is empty.")
                continue
            print(f"  Processing well_country: {well_country}")

            try:
                exposed = pop_est.est_exposed_pop(
                    pop_path=pop_path,
                    hazard_specific=False,
                    hazards=hazards_group,
                    admin_units=admin_units
                )
            except KeyError as e:
                print(f"    Skipping well_country {well_country}: missing key {e}.")
                continue

            # Skip if exposed is empty
            if exposed is None or getattr(exposed, "empty", True):
                print(f"    Skipping well_country {well_country}: exposure result is empty.")
                continue

            exposed["country"] = country
            exposed["well_country"] = well_country
            exposed_list.append(exposed)

    else:
            try:
                exposed = pop_est.est_exposed_pop(
                    pop_path=pop_path,
                    hazard_specific=False,
                    hazards=hazards,
                    admin_units=admin_units
                )
            except KeyError as e:
                print(f"  Skipping {country}: missing key {e}.")
                exposed = None
            except Exception as e:
                print(f"  Skipping {country}: unexpected error {e}.")
                exposed = None

            if exposed is not None and not getattr(exposed, "empty", True):
                exposed["country"] = country
                exposed_list.append(exposed)
            else:
                print(f"  Skipping {country}: exposure result is empty or invalid.")

    try:
        pop_df = pop_est.est_pop(pop_path=pop_path, admin_units=admin_units)
        pop_df["country"] = country
        pop_list.append(pop_df)
    except Exception as e:
        print(f"  Skipping population estimate for {country}: {e}")

all_exposed_df = pd.concat(exposed_list, axis=0, ignore_index=True)
all_pop_df = pd.concat(pop_list, axis=0, ignore_index=True)

# Ensure well_country is present for merge
if "well_country" not in all_exposed_df.columns:
    all_exposed_df["well_country"] = np.nan

# Merge on ID_admin_unit, country (well_country will be NaN if not present)
merge_cols = ["ID_admin_unit", "country"]
if "well_country" in all_exposed_df.columns:
    merge_cols.append("well_country")

exposed_cols = ["exposed_1km", "exposed_3km"]
if "exposed_5km" in all_exposed_df.columns:
    exposed_cols.append("exposed_5km")

combined_df = all_pop_df.merge(
    all_exposed_df[merge_cols + exposed_cols],
    on=["ID_admin_unit", "country"],  # merge on ID_admin_unit and country
    how="left"
)

# Add percent exposed columns
combined_df["pct_exposed_1km"] = combined_df["exposed_1km"] / combined_df["population"]
combined_df["pct_exposed_3km"] = combined_df["exposed_3km"] / combined_df["population"]
if "exposed_5km" in combined_df.columns:
    combined_df["pct_exposed_5km"] = combined_df["exposed_5km"] / combined_df["population"]

# Optionally, handle division by zero or NaN
combined_df["pct_exposed_1km"] = combined_df["pct_exposed_1km"].replace([np.inf, -np.inf], np.nan)
combined_df["pct_exposed_3km"] = combined_df["pct_exposed_3km"].replace([np.inf, -np.inf], np.nan)
if "pct_exposed_5km" in combined_df.columns:
    combined_df["pct_exposed_5km"] = combined_df["pct_exposed_5km"].replace([np.inf, -np.inf], np.nan)

# Group and sum by country and well_country if present
group_cols = ["country", "well_country"]

summary_df = combined_df.groupby(
    group_cols, as_index=False
)[exposed_cols + ["population"]].sum(numeric_only=True)

# Rename columns for clarity
summary_df = summary_df.rename(columns={
    "exposed_1km": "out_of_country_exposed_1km",
    "exposed_3km": "out_of_country_exposed_3km",
    "exposed_5km": "out_of_country_exposed_5km" if "exposed_5km" in summary_df.columns else None
})

print(summary_df.head())

output_path = results_dir / "ogd_country_pop_exposed_out_of_country.csv"
summary_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

In [ ]:
# Function step 4: Estimating population affected by oil and gas wells by adm2 - offshore wells

## Apply Popexposure function
results = {}

# Find total num ppl residing 
exposed_list = []
pop_list = []

# Loop through each country
for country in country_list:
    print(f"Processing {country}...")

    # types of wells
    hazard_paths = ogim_dir / f"OGIM_{country}_offshore_modis.parquet"

    # country parquet
    admin_units_path = country_dir / f"{country}_adm2.parquet"

    # Check if both files exist
    if not hazard_paths.exists():
        print(f"  Skipping {country}: offshore file not found.")
        continue
    if not admin_units_path.exists():
        print(f"  Skipping {country}: admin2 file not found.")
        continue

    # Prepare admin units data
    admin_units = pop_est.prep_data(admin_units_path, geo_type='admin_unit')

    for hazard_path in [hazard_paths]:
        try:
            # Prepare hazard data for this year
            hazards = pop_est.prep_data(hazard_path, geo_type='hazard')

            exposed = pop_est.est_exposed_pop(
                pop_path=pop_path,
                hazard_specific=False,  # set to True if you want per-hazard results
                hazards=hazards,
                admin_units=admin_units
            )

        except KeyError as e:
            print(f"  Skipping {country}: missing key {e}.")
            continue
        except Exception as e:
            print(f"  Skipping {country}: unexpected error {e}")
            continue

        # Skip if no result
        if exposed is None or getattr(exposed, "empty", True):
            print(f"  Skipping {country}: exposure result is empty or invalid.")
            continue

        exposed["country"] = country  
        exposed_list.append(exposed)

# Estimate total population per spatial unit for this country
    pop_df = pop_est.est_pop(pop_path=pop_path, admin_units=admin_units)
    pop_df["country"] = country  # Optionally add country column for tracking
    pop_list.append(pop_df)      # <-- Collect each country's pop_df

# Combine all countries
all_exposed_df = pd.concat(exposed_list, axis=0, ignore_index=True)
#all_exposed_df = all_exposed_df.groupby(["ID_spatial_unit", "country"], as_index=False).sum(numeric_only=True)
all_pop_df = pd.concat(pop_list, axis=0, ignore_index=True)

# Merge exposed and total population on ID_spatial_unit and country
combined_df = all_pop_df.merge(
    all_exposed_df[["ID_admin_unit", "exposed_1km", "exposed_3km", "exposed_5km", "country"]],
    on=["ID_admin_unit", "country"],
    how="left"
)

# Add percent exposed columns
combined_df["pct_exposed_1km"] = combined_df["exposed_1km"] / combined_df["population"]
combined_df["pct_exposed_3km"] = combined_df["exposed_3km"] / combined_df["population"]
combined_df["pct_exposed_5km"] = combined_df["exposed_5km"] / combined_df["population"]

# Optionally, handle division by zero or NaN
combined_df["pct_exposed_1km"] = combined_df["pct_exposed_1km"].replace([np.inf, -np.inf], np.nan)
combined_df["pct_exposed_3km"] = combined_df["pct_exposed_3km"].replace([np.inf, -np.inf], np.nan)
combined_df["pct_exposed_5km"] = combined_df["pct_exposed_5km"].replace([np.inf, -np.inf], np.nan)

# Group and sum by country 
group_cols = ["country"]
if "well_country" in combined_df.columns:
    group_cols.append("well_country")

summary_df = combined_df.groupby(
    group_cols, as_index=False
)[["exposed_1km", "exposed_3km", "exposed_5km", "population"]].sum(numeric_only=True)

summary_df = summary_df.rename(columns={
    "exposed_1km": "offshore_exposed_1km",
    "exposed_3km": "offshore_exposed_3km",
    "exposed_5km": "offshore_exposed_5km"
})

print(summary_df.head())

# Export summary_df as CSV to the results folder
output_path = results_dir / "ogd_country_pop_exposed_offshore.csv"
summary_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")


In [ ]:
## Function step 5: Estimating total area exposed by country and by adm2
pop_path = pathlib.Path("/Users/larasch/Library/CloudStorage/GoogleDrive-laranschwarz@gmail.com/My Drive/EarthEngineExports/MOD44W_2015_land_mask_clipped_anyoverlap.tif")

# Find total num ppl residing 
exposed_list = []
pop_list = []  

# Loop through each country
for country in country_list:
    print(f"Processing {country}...")

    # types of wells
    hazard_paths = ogim_dir / f"OGIM_{country}.parquet"
    
    # country parquet
    admin_units_path = country_dir / f"{country}_adm2.parquet"

    # Prepare admin units data
    admin_units = pop_est.prep_data(admin_units_path, geo_type='admin_unit')

    
    for hazard_path in [hazard_paths]:
        try:
            # Prepare hazard data for this year
            hazards = pop_est.prep_data(hazard_path, geo_type='hazard')

            exposed = pop_est.est_exposed_pop(
                pop_path=pop_path,
                hazard_specific=False,  # set to True if you want per-hazard results
                hazards=hazards,
                admin_units=admin_units
            )

        except KeyError as e:
            print(f"  Skipping {country}: missing key {e}.")
            continue
        except Exception as e:
            print(f"  Skipping {country}: unexpected error {e}")
            continue

        # Skip if no result
        if exposed is None or getattr(exposed, "empty", True):
            print(f"  Skipping {country}: exposure result is empty or invalid.")
            continue

        exposed["country"] = country  
        exposed_list.append(exposed)

# Estimate total population per spatial unit for this country
    pop_df = pop_est.est_pop(pop_path=pop_path, admin_units=admin_units)
    pop_df["country"] = country  # Optionally add country column for tracking
    pop_list.append(pop_df)      # <-- Collect each country's pop_df

# Combine all countries
all_exposed_df = pd.concat(exposed_list, axis=0, ignore_index=True)
all_pop_df = pd.concat(pop_list, axis=0, ignore_index=True)

# Merge exposed and total population on ID_spatial_unit and country
combined_df = all_pop_df.merge(
    all_exposed_df[["ID_admin_unit", "country",   "exposed_1km", "exposed_3km","exposed_5km"]],
    on=["ID_admin_unit", "country"],
    how="left"
)
# Add percent exposed columns
for dist in ["1km", "3km", "5km"]:
    combined_df[f"pct_exposed_{dist}"] = combined_df[f"exposed_{dist}"] / combined_df["population"]
    combined_df[f"pct_exposed_{dist}"] = combined_df[f"pct_exposed_{dist}"].replace([np.inf, -np.inf], np.nan)

# Export summary_df as CSV to the results folder
#output_path = results_dir / "ogd_land_area_exposed_adm2.csv"
#combined_df.to_csv(output_path, index=False)
#print(f"Saved: {output_path}")

## Export for land area estimates at country level

country_summary = (
    combined_df.groupby("country", as_index=False)
    .agg({
        "exposed_1km": "sum",
        "exposed_3km": "sum",
        "exposed_5km": "sum",
        "population": "sum"
    })
)

# Compute country-level percentages
for dist in ["1km", "3km", "5km"]:
    country_summary[f"pct_exposed_{dist}"] = (
        country_summary[f"exposed_{dist}"] / country_summary["population"]
    )

# Handle NaNs and infinite values
country_summary.replace([np.inf, -np.inf], np.nan, inplace=True)

# Export to CSV
#country_output_path = results_dir / "ogd_land_exposed_country.csv"
#country_summary.to_csv(country_output_path, index=False)
#print(f"Saved country-level summary: {country_output_path}")

In [6]:

## testing testing

BUFFER_DIST = 5000  # 5 km buffer

country_list = [ 'belgium', 'united_states']

def detect_state_column(gdf):
    candidates = ["ADM1_NAME", "state", "state_name", "NAME_1", "adm1_name"]
    for c in candidates:
        if c in gdf.columns:
            return c
    raise ValueError("No state / ADM1 column found in U.S. ADM2 dataset.")


# Function step 6: Estimating ogd well-weighted population exposure 
results = {}

exposed_list = []
pop_list = []

for country in country_list:
    print(f"Processing {country}...")

    hazard_paths = ogim_dir / f"OGIM_{country}.parquet"
    admin_units_path = country_dir / f"{country}_adm2.parquet"

    # ---------- SPECIAL HANDLING FOR UNITED STATES ----------
    if country == "United_States":

        print("  Splitting U.S. by state with 5 km border inclusion...")

        # Load ADM2 once
        us_admin = pop_est.prep_data(admin_units_path, geo_type="admin_unit")
        state_col = detect_state_column(us_admin)
        states = sorted(us_admin[state_col].unique())

        # Load full hazard dataset
        us_haz = pop_est.prep_data(hazard_path, geo_type="hazard")

        # Merge ADM2 to create a full-state geometry (ADM1)
        print("  Creating state boundaries...")
        state_boundaries = us_admin.dissolve(by=state_col)[["geometry"]]

        # Create 5 km buffer for each state
        print("  Building 5 km buffers for state borders...")
        state_boundaries["geometry"] = state_boundaries.geometry.buffer(BUFFER_DIST)

        # Spatial join wells to buffered state polygons
        print("  Assigning wells to states (within 5 km buffer)...")
        hazards_with_state = gpd.sjoin(
            us_haz,
            state_boundaries,
            how="left",
            predicate="intersects"
        ).rename(columns={state_col: "state"})

        # Loop through states
        for st in states:
            print(f"    Processing state: {st}")

            # Wells inside state or within 5 km of border
            haz_st = hazards_with_state[hazards_with_state["state"] == st]

            if haz_st.empty:
                print(f"      No hazards in or near {st}; skipping.")
                continue

            # ADM2 units for this state only
            admin_st = us_admin[us_admin[state_col] == st]

            try:
                exposed = pop_est.est_exposed_pop(
                    pop_path=pop_path,
                    hazard_specific=True,
                    hazards=haz_st,
                    admin_units=admin_st
                )
            except Exception as e:
                print(f"      Skipping {st}: {e}")
                continue

            if exposed is None or getattr(exposed, "empty", True):
                print(f"      Empty result for {st}.")
                continue

            exposed["country"] = country
            exposed["state"] = st
            exposed_list.append(exposed)

            # population for this state's ADM2 units
            pop_df = pop_est.est_pop(pop_path=pop_path, admin_units=admin_st)
            pop_df["country"] = country
            pop_df["state"] = st
            pop_list.append(pop_df)

        continue




    # Prepare admin units data
    admin_units = pop_est.prep_data(admin_units_path, geo_type='admin_unit')

    for hazard_path in [hazard_paths]:
        try:
            # Prepare hazard data for this year
            hazards = pop_est.prep_data(hazard_path, geo_type='hazard')

            exposed = pop_est.est_exposed_pop(
                pop_path=pop_path,
                hazard_specific=True,  # set to True if you want per-hazard results
                hazards=hazards,
                admin_units=admin_units
            )

        except KeyError as e:
            print(f"  Skipping {country}: missing key {e}.")
            continue
        except Exception as e:
            print(f"  Skipping {country}: unexpected error {e}")
            continue

        # Skip if no result
        if exposed is None or getattr(exposed, "empty", True):
            print(f"  Skipping {country}: exposure result is empty or invalid.")
            continue

        exposed["country"] = country  
        exposed_list.append(exposed)    

# Estimate total population per spatial unit for this country
    pop_df = pop_est.est_pop(pop_path=pop_path, admin_units=admin_units)
    pop_df["country"] = country  # Optionally add country column for tracking
    pop_list.append(pop_df)      # <-- Collect each country's pop_df

# Combine all countries
all_exposed_df = pd.concat(exposed_list, axis=0, ignore_index=True)
all_pop_df = pd.concat(pop_list, axis=0, ignore_index=True)

# Rename population column in pop_df to avoid confusion
all_pop_df = all_pop_df.rename(columns={"population": "total_population"})

# Merge on ID_admin_unit and country
combined_df = all_exposed_df.merge(
    all_pop_df[["ID_admin_unit", "country", "total_population"]],
    on=["ID_admin_unit", "country"],
    how="left"
)

# Group by country and ID_admin_unit, summing the hazard-specific and total populations
summary_df = combined_df.groupby(["country", "ID_admin_unit"], as_index=False).agg({
    "exposed_population": "sum",
    "total_population": "first"  # Use 'first' because total pop is the same per admin unit
})

print(summary_df.head())

# Save output
#output_path = results_dir / "ogd_country_pop_exposed_well_weighted.csv"
##combined_df.to_csv(output_path, index=False)
#print(f"Saved: {output_path}")

Processing belgium...


/opt/anaconda3/envs/pop_exp/lib/python3.11/site-packages/osgeo/gdal.py:311: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Processing united_states...


: 

In [4]:
combined_df

,ID_hazard,ID_admin_unit,exposed_1km,exposed_3km,exposed_5km,country,land_area
0,1,6016009,0,145,515,argentina,227407
1,1,16011009,50,306,739,argentina,210004
2,10,5010009,0,7,134,argentina,44716
3,10,18017009,50,443,1120,argentina,189567
4,100,18017009,50,451,1255,argentina,189567
...,...,...,...,...,...,...,...
1123879,9995,8002239,50,451,1255,venezuela,121236
1123880,9996,8002239,50,451,1255,venezuela,121236
1123881,9997,8002239,50,451,1255,venezuela,121236
1123882,9998,8002239,50,451,1255,venezuela,121236


In [5]:
# Function step 6: Estimating ogd well-weighted land exposure 

country_list = [
    "argentina", "australia", "austria", "belgium", "bolivia", "brazil", "canada", "chile", 
    "colombia", "denmark", "ecuador", "ethiopia", "france", "germany", "guatemala", "ireland", "italy", 
    "libya", "mexico", "mozambique", "netherlands", "new_zealand", "norway", 
    "papua_new_guinea", "paraguay", "peru", "saudi_arabia", "south_sudan", "sudan", 
    "united_arab_emirates", "united_kingdom", "venezuela"
]
## Apply THE function
results = {}

# Find total num ppl residing 
exposed_list = []
pop_list = []

# Loop through each country
for country in country_list:
    print(f"Processing {country}...")

    # types of wells
    hazard_paths = ogim_dir / f"OGIM_{country}.parquet"
    
    # country parquet
    #admin_units_path = country_dir / f"{country}.parquet"
    admin_units_path = country_dir / f"{country}_adm2.parquet"

    # Prepare admin units data
    admin_units = pop_est.prep_data(admin_units_path, geo_type='admin_unit')

    for hazard_path in [hazard_paths]:
        try:
            # Prepare hazard data for this year
            hazards = pop_est.prep_data(hazard_path, geo_type='hazard')

            exposed = pop_est.est_exposed_pop(
                pop_path=land_path,
                hazard_specific=True,  # set to True if you want per-hazard results
                hazards=hazards,
                admin_units=admin_units
            )

        except KeyError as e:
            print(f"  Skipping {country}: missing key {e}.")
            continue
        except Exception as e:
            print(f"  Skipping {country}: unexpected error {e}")
            continue

        # Skip if no result
        if exposed is None or getattr(exposed, "empty", True):
            print(f"  Skipping {country}: exposure result is empty or invalid.")
            continue

        exposed["country"] = country  
        exposed_list.append(exposed)    

# Estimate total population per spatial unit for this country
    pop_df = pop_est.est_pop(pop_path=land_path, admin_units=admin_units)
    pop_df["country"] = country  # Optionally add country column for tracking
    pop_list.append(pop_df)      # <-- Collect each country's pop_df

# Combine all countries
all_exposed_df = pd.concat(exposed_list, axis=0, ignore_index=True)
all_pop_df = pd.concat(pop_list, axis=0, ignore_index=True)

# Rename population column in pop_df to avoid confusion
all_pop_df = all_pop_df.rename(columns={"population": "land_area"})

# Merge on ID_admin_unit and country
combined_df = all_exposed_df.merge(
    all_pop_df[["ID_admin_unit", "country", "land_area"]],
    on=["ID_admin_unit", "country"],
    how="left"
)

# Group by country and ID_admin_unit, summing the hazard-specific and total populations
summary_df = combined_df.groupby(["country", "ID_admin_unit"], as_index=False).agg({
    "exposed_1km": "sum",
    "exposed_3km": "sum",
    "exposed_5km": "sum",
    "land_area": "first"  # Use 'first' because total pop is the same per admin unit
})

print(summary_df.head())

# Save output
output_path = results_dir / "ogd_country_land_exposed_well_weighted.csv"
combined_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Processing argentina...
Processing australia...
Processing austria...
Processing belgium...
Processing bolivia...
Processing brazil...
Processing canada...
Processing chile...
Processing colombia...
Processing denmark...
Processing ecuador...
Processing ethiopia...
Processing france...
Processing germany...
Processing guatemala...
  Skipping guatemala: unexpected error [Errno 2] Failed to open local file '/Users/larasch/Library/CloudStorage/OneDrive-SharedLibraries-UW/casey_cohort - Documents/data/environmental/oil_and_gas/raw_data/ogim_by_country/OGIM_guatemala.parquet'. Detail: [errno 2] No such file or directory
Processing ireland...
Processing italy...
Processing libya...
Processing mexico...
Processing mozambique...
Processing netherlands...
Processing new_zealand...
Processing norway...
  Skipping norway: missing key 'geometry'.
Processing papua_new_guinea...
Processing paraguay...
Processing peru...
Processing saudi_arabia...
Processing south_sudan...
Processing sudan...
Process

In [ ]:
#Function step 7: Estimating average GRDI by adm2 - well-weighted

# Define the base paths for GRDI data, this is not population data but has to be named that way for function
pop_path = share_path / "social_including_census/raw_data/GRDI_v1/ciesin_nasa_povmap-grdi-v1.tif"

# Find total num ppl residing 
exposed_list = []
pop_list = []  

# Loop through each country
for country in country_list:
    print(f"Processing {country}...")

    # types of wells
    hazard_paths = ogim_dir / f"OGIM_{country}.parquet"
    
    # country parquet
    admin_units_path = country_dir / f"{country}_adm2.parquet"

    # Prepare admin units data
    admin_units = pop_est.prep_data(admin_units_path, geo_type='admin_unit')

    for hazard_path in [hazard_paths]:
        try:
            # Prepare hazard data for this year
            hazards = pop_est.prep_data(hazard_path, geo_type='hazard')

            exposed = pop_est.est_exposed_pop(
                pop_path=pop_path,
                hazard_specific=False,  # set to True if you want per-hazard results
                hazards=hazards,
                admin_units=admin_units,
                stat= "mean"    # get average GRDI score for admin 2 unit
            )

        except KeyError as e:
            print(f"  Skipping {country}: missing key {e}.")
            continue
        except Exception as e:
            print(f"  Skipping {country}: unexpected error {e}")
            continue

        # Skip if no result
        if exposed is None or getattr(exposed, "empty", True):
            print(f"  Skipping {country}: exposure result is empty or invalid.")
            continue

        exposed["country"] = country  
        exposed_list.append(exposed)

# Estimate total population per spatial unit for this country
   # pop_df = pop_est.pop(pop_path=pop_path, spatial_units=admin_units)
   # pop_df["country"] = country  # Optionally add country column for tracking
    #pop_list.append(pop_df)      # <-- Collect each country's pop_df

# Combine all countries
all_exposed_df = pd.concat(exposed_list, axis=0, ignore_index=True)
#all_pop_df = pd.concat(pop_list, axis=0, ignore_index=True)

all_exposed_df = all_exposed_df.rename(columns={
    'exposed_1km': 'GRDI_1km',
    'exposed_3km': 'GRDI_3km',
    'exposed_5km': 'GRDI_5km'
})

# Save output
# Save output
output_path = results_dir / "ogd_adm2_avg_GRDI.csv"
all_exposed_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")